# ETL del Euríbor

Este *notebook* accede a los datos del Banco de España a través de una API. Dicha API se adjunta en este código, pero su uso queda restringido, pudiendo ser utilizada solo con fines correctivos por parte del tutor.

In [1]:
# Importar librerías
import os, sys, requests
import time as time_module
from datetime import datetime, date, timedelta, time
import numpy as np
import pandas as pd

In [2]:
def fetch_bde_series(series_codes, rango="12M", language="es"):
    """
    Obtiene datos de series temporales del Banco de España mediante su API REST.
    Descarga una (o múltiples series económicas) y las estructura en DataFrames
    con fechas y valores numéricos organizados por código de serie.
    
    Args:
        - series_codes (list): Lista de códigos de series del BDE (ej: ["D_DNBAF172"])
        - rango (str, optional): Período de datos ("3M", "12M", "36M"). Default "12M"
        - language (str, optional): Idioma de respuesta ("es", "en"). Default "es"
    
    Returns:
        - dict: Diccionario donde cada clave es un código de serie y el valor es un
             pd.DataFrame con columnas "fecha" (date) y descripción+símbolo (float/None)
             ordenado cronológicamente
    
    Raises:
        - requests.HTTPError: Si la API devuelve error HTTP
    """
    base_url = "https://app.bde.es/bierest/resources/srdatosapp/listaSeries"
    
    # Construir parámetro "series" como cadena comma-separated
    series_param = ",".join(series_codes)
    params = {
        "idioma": language,
        "series": series_param,
        "rango": rango
    }
    resp = requests.get(base_url, params=params)
    resp.raise_for_status()  # Lanzará error si HTTP != 200
    data = resp.json()
    # data es una lista de objetos, uno por serie en series_codes
    
    result = {}
    for entry in data:
        code = entry.get("serie")
        description = entry.get("descripcionCorta")
        symbol = entry.get("simbolo")
        dates = entry.get("fechas", [])
        values = entry.get("valores", [])
        
        # Convertir a DataFrame
        df = pd.DataFrame({
            "fecha": pd.to_datetime(dates),
            description + symbol: values
        })
        
        # Ordenar por Fecha (aunque la API normalmente ya envía ordenado)
        df["fecha"] = df["fecha"].dt.date
        df = df.sort_values("fecha").drop_duplicates().reset_index(drop=True) # Ordenar por fecha
        result[code] = df
        
    return result

In [3]:
# Código del euríbor diario a 12 meses
series_codes = ["D_DNBAF172"]

# Obtener últimos 36 meses
dfs = fetch_bde_series(series_codes, rango="36M", language="es")
df_euribor = dfs.get("D_DNBAF172")
df_euribor.head()

,fecha,Euribor a 12 meses%
0,2022-11-03,2.735
1,2022-11-04,2.794
2,2022-11-07,2.820
3,2022-11-08,2.846
4,2022-11-09,2.874


In [4]:
project_root = os.getcwd()
while project_root != os.path.dirname(project_root) and not os.path.exists(os.path.join(project_root, "configs", "config.yaml")):
    project_root = os.path.dirname(project_root)
store_path = os.path.join(project_root, "data", "raw")

In [5]:
# Renombrar columnas
df_euribor.rename(columns={"Euribor a 12 meses%": "euribor_a_12_meses%"}, inplace=True)

# Filtrar fechas
mask = (date(2024, 1, 1) <= df_euribor["fecha"]) & (df_euribor["fecha"] <= date(2025, 5, 25))
df_euribor = df_euribor[mask]

# Guardar dataframe
df_euribor.to_csv(os.path.join(store_path, "euribor.csv"), index=False)